<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/claude%2Fkind-volta-wlb7e5/vllm_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vLLM on Colab L4 / A100 / G4

Installs vLLM (the packages are cached on Google Drive), serves a model that fits the GPU with tool calling and an OpenAI-compatible API, and connects the DeepSeek Harness (dsh) agent front end.

Runtime -> Change runtime type -> **L4**, **A100** or **G4 GPU**. vLLM serves safetensors checkpoints (bf16, FP8, NVFP4, AWQ...), not GGUF files; for GGUF use `llama_cpp_gpu.ipynb`. A T4 is not supported by recent vLLM builds; use the llama.cpp notebook there.

## 1. Detect the GPU

In [ ]:
import os, re, subprocess
os.environ["PATH"] += ":/usr/local/cuda/bin"

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if smi.returncode or not smi.stdout.strip():
    raise RuntimeError("No NVIDIA GPU found. Runtime -> Change runtime type -> choose an L4, A100 or G4 GPU.")
GPU_NAME, mem_mib, cc = [x.strip() for x in smi.stdout.strip().splitlines()[0].split(",")]
VRAM_GB   = int(float(mem_mib)) / 1024
CUDA_ARCH = cc.replace(".", "")                      # 8.0 -> 80, 8.9 -> 89, 12.0 -> 120
TIER = "xl" if VRAM_GB >= 80 else "l" if VRAM_GB >= 36 else "m" if VRAM_GB >= 20 else "s"
print(f"{GPU_NAME}: {VRAM_GB:.0f} GB VRAM, compute capability {cc} (sm_{CUDA_ARCH}), size tier '{TIER}'")
!free -h | head -2
!df -h /content | tail -1

## 2. Config (the model is picked from the GPU's memory; override any value)

In [ ]:
# Qwen3.8-27B needs vLLM 0.17 or newer (pip installs the latest). Checkpoint sizes: bf16 ~55.6 GB, FP8 ~30.9 GB, NVFP4 (Blackwell) ~21.6 GB.
PRESETS = {
    # 80+ GB: G4 (96 GB), A100 80 GB. FP8 leaves lots of room for context; use "Qwen/Qwen3.8-27B" for full bf16.
    "xl": dict(model="Qwen/Qwen3.8-27B-FP8", len=65536, util=0.90, tools="qwen3_coder", reasoning="qwen3"),
    # 36-79 GB: A100 40 GB. 30.9 GB of weights, so the context stays short.
    "l":  dict(model="Qwen/Qwen3.8-27B-FP8", len=16384, util=0.92, tools="qwen3_coder", reasoning="qwen3"),
    # 20-35 GB: L4 (24 GB). The 27B FP8 does not fit; 14B FP8 (16.3 GB) does.
    "m":  dict(model="Qwen/Qwen3-14B-FP8",   len=16384, util=0.90, tools="hermes",      reasoning="qwen3"),
    # under 20 GB: a small model, if your vLLM build supports the GPU at all
    "s":  dict(model="Qwen/Qwen3-4B",        len=8192,  util=0.90, tools="hermes",      reasoning="qwen3"),
}

# Override to choose your own (any Hugging Face safetensors model vLLM supports).
MODEL_ID         = None
MAX_MODEL_LEN    = None
TOOL_PARSER      = None   # vLLM tool-call parser; Qwen3.5/3.8 use qwen3_coder (some builds also offer qwen3_xml), Qwen3 uses hermes
REASONING_PARSER = None   # "qwen3" separates the thinking text from the answer
REFRESH_WHEELS   = False  # True = re-download the packages and re-save them on Drive

p = PRESETS[TIER]
MODEL_ID         = MODEL_ID or p["model"]
MAX_MODEL_LEN    = MAX_MODEL_LEN or p["len"]
TOOL_PARSER      = TOOL_PARSER or p["tools"]
REASONING_PARSER = REASONING_PARSER or p["reasoning"]
GPU_UTIL         = p["util"]
print("Model:", MODEL_ID, "| context:", MAX_MODEL_LEN, "| tool parser:", TOOL_PARSER)

## 3. Keep the Hugging Face cache on Google Drive (the model is downloaded once and reused)

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
HF_CACHE = "/content/drive/MyDrive/llm_models/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE          # inherited by the vllm server started below
print("Hugging Face cache:", HF_CACHE)

## 4. Install vLLM (the packages are saved on Google Drive after the first install)

In [ ]:
import sys, os, glob, subprocess

WHEELS = f"/content/drive/MyDrive/llm_models/builds/vllm_gpu_py{sys.version_info.major}{sys.version_info.minor}"
PKGS = ["vllm"]

def pip(*args):
    r = subprocess.run([sys.executable, "-m", "pip", *args], capture_output=True, text=True)
    open("/content/pip_install.log", "a").write(r.stdout + r.stderr)
    if r.returncode:
        raise RuntimeError("pip failed:\n" + (r.stdout + r.stderr)[-3000:])

open("/content/pip_install.log", "w").close()
cached = bool(glob.glob(f"{WHEELS}/*.whl")) and not REFRESH_WHEELS
if cached:
    print("Installing from the wheels saved on Drive...")
    try:
        pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
    except RuntimeError as e:
        print("Cached install failed, rebuilding the wheel cache:\n", str(e)[:500])
        cached = False
if not cached:
    print("Downloading packages and saving them to Drive (slow, one time; several GB including PyTorch)...")
    os.makedirs(WHEELS, exist_ok=True)
    pip("wheel", "-q", "-w", WHEELS, *PKGS)
    pip("install", "-q", "--no-index", "--find-links", WHEELS, *PKGS)
print("Done. Wheels:", len(glob.glob(f"{WHEELS}/*.whl")))
print(subprocess.run([sys.executable, "-m", "pip", "show", "vllm"], capture_output=True, text=True).stdout.split("\n")[1])

## 5. Start the server (the first start downloads the model, then loads it and compiles kernels, which can take several minutes)

In [ ]:
import subprocess, time, requests, secrets

API_KEY = secrets.token_urlsafe(24)   # required by every request, including through the public tunnel

cmd = ["vllm", "serve", MODEL_ID,
       "--max-model-len", str(MAX_MODEL_LEN),
       "--gpu-memory-utilization", str(GPU_UTIL),
       "--max-num-seqs", "16",
       "--enable-auto-tool-choice", "--tool-call-parser", TOOL_PARSER,
       "--host", "127.0.0.1", "--port", "8000", "--api-key", API_KEY]
if REASONING_PARSER:
    cmd += ["--reasoning-parser", REASONING_PARSER]

server = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(900):                      # up to ~30 min
    if server.poll() is not None:
        print("server exited:\n", open("/content/vllm.log").read()[-5000:]); break
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("server ready"); break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(open("/content/vllm.log").read()[-4000:])

## 6. Quick test from inside the notebook

In [ ]:
def chat(prompt, max_tokens=512, temperature=0.7):
    t0 = time.time()
    r = requests.post("http://127.0.0.1:8000/v1/chat/completions",
                      headers={"Authorization": f"Bearer {API_KEY}"},
                      json={"model": MODEL_ID, "messages": [{"role": "user", "content": prompt}],
                            "max_tokens": max_tokens, "temperature": temperature}, timeout=900)
    r.raise_for_status()
    j = r.json(); m = j["choices"][0]["message"]
    n = j.get("usage", {}).get("completion_tokens")
    if n: print(f"[{n} tokens, {n / (time.time() - t0):.1f} tokens/s including prompt processing]")
    thinking = m.get("reasoning_content") or m.get("reasoning")
    return (("[thinking]\n" + thinking + "\n[answer]\n") if thinking else "") + (m.get("content") or "")

print(chat("Explain what vLLM does in two sentences."))

## 7. Use the OpenAI-compatible API from outside Colab

This opens a free Cloudflare tunnel to the vLLM server. The URL is public but every request needs the API key. Anyone with both can use your GPU, so don't share them.

In [ ]:
import subprocess, re, time, os, urllib.request

CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)

def open_tunnel(port):
    """Expose a local port through a free Cloudflare quick tunnel and return (process, public URL)."""
    log_path = f"/content/cloudflared_{port}.log"
    proc = subprocess.Popen([CF, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
                            stdout=open(log_path, "w"), stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return proc, m.group(0)
        time.sleep(1)
    raise RuntimeError("No tunnel URL yet:\n" + open(log_path).read()[-2000:])

In [ ]:
tunnel, PUBLIC_URL = open_tunnel(8000)
print("API    :", PUBLIC_URL + "/v1")
print("API key:", API_KEY)
print("Model  :", MODEL_ID)
print(f"""
from openai import OpenAI
client = OpenAI(base_url="{PUBLIC_URL}/v1", api_key="{API_KEY}")
print(client.chat.completions.create(model="{MODEL_ID}", messages=[{{"role": "user", "content": "Hello"}}]).choices[0].message.content)
""")

## 8. DeepSeek Harness (dsh) front end

[dsh](https://github.com/deepseek-ai/deepseek-harness) is DeepSeek's open-source agent harness: a coding agent with a local web UI that works with any OpenAI-compatible server. This points it at the server started above and opens it through a tunnel.

- **Security:** dsh's agent can run shell commands and edit files on this Colab VM, and **Google Drive is mounted** here. Keep dsh's tool-approval prompts on, and don't approve anything touching `/content/drive` unless you mean to. The tunnel URL is public; dsh protects itself with a random access token that is part of the link printed below, so share that link with nobody.
- dsh needs Node 22.19+, which this installs (Colab's default is older).
- Agent tool calling only works if the server has the right flags (`--jinja` for llama.cpp, `--enable-auto-tool-choice --tool-call-parser ...` for vLLM). The server cell above sets them.
- An agent needs a long context (at least 16384 tokens); the context size is set from the GPU in the config cell.

In [ ]:
import os, re, subprocess, urllib.request, time, socket, textwrap

# ---- what to connect dsh to ----
BACKEND  = "http://127.0.0.1:8000"            # the OpenAI-compatible server started above
PROVIDER = "vllm"
MODEL    = MODEL_ID
CTX      = MAX_MODEL_LEN                  # must match the server's context size
MAXTOK   = min(4096, CTX // 2)

# ---- Node 22 (dsh needs 22.19+ or 24+) ----
NODE_DIR = "/content/node"
if not os.path.exists(f"{NODE_DIR}/bin/node"):
    base = "https://nodejs.org/dist/latest-v22.x/"
    fn = re.search(r"node-v22[\d.]+-linux-x64\.tar\.xz", urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()).group(0)
    urllib.request.urlretrieve(base + fn, "/content/node.tar.xz")
    os.makedirs(NODE_DIR, exist_ok=True)
    subprocess.run(["tar", "-xJf", "/content/node.tar.xz", "-C", NODE_DIR, "--strip-components=1"], check=True)
NPM_CACHE = "/content/drive/MyDrive/llm_models/builds/npm_cache"   # on Drive so dsh is not re-downloaded every session
os.makedirs(NPM_CACHE, exist_ok=True)
ENV = dict(os.environ, PATH=f"{NODE_DIR}/bin:" + os.environ["PATH"], npm_config_cache=NPM_CACHE, LOCAL_LLM_API_KEY=API_KEY)
print("node", subprocess.run(["node", "--version"], env=ENV, capture_output=True, text=True).stdout.strip())

# ---- dsh provider config ----
os.makedirs(os.path.expanduser("~/.dsh"), exist_ok=True)
open(os.path.expanduser("~/.dsh/settings.yaml"), "w").write(f"""llm-pi-ai:
  providers:
    {PROVIDER}:
      displayName: {PROVIDER}
      apiKeyEnv: LOCAL_LLM_API_KEY
      api: openai-completions
      baseURL: {BACKEND}/v1
      compat:
        supportsDeveloperRole: false
        maxTokensField: max_tokens
      models:
        - id: {MODEL}
          name: {MODEL}
          contextWindow: {CTX}
          maxTokens: {MAXTOK}
agent-default-model:
  provider: {PROVIDER}
  model: {MODEL}
""")

# ---- start dsh (web UI listens on 127.0.0.1:3080) ----
os.makedirs("/content/workspace", exist_ok=True)
dsh = subprocess.Popen(["npx", "-y", "@deepseek-ai/dsh", "web", "--no-open"], cwd="/content/workspace", env=ENV,
                       stdout=open("/content/dsh.log", "w"), stderr=subprocess.STDOUT)
def wait_for_port(port, proc, seconds=180):
    for _ in range(seconds):
        if proc.poll() is not None:
            return False
        try:
            socket.create_connection(("127.0.0.1", port), timeout=1).close()
            return True
        except OSError:
            time.sleep(1)
    return False

if not wait_for_port(3080, dsh):
    print(open("/content/dsh.log").read()[-3000:])
    raise RuntimeError("dsh did not start; see the log above")

# ---- publish through the tunnel; dsh protects itself with a random token in the link ----
token = re.search(r"token=([\w-]+)", open("/content/dsh.log").read()).group(1)
dsh_tunnel, DSH_URL = open_tunnel(3080)
print("Open this link (it contains the access token; treat it like a password):")
print(f"{DSH_URL}/?token={token}")
print("If the model is not preselected, pick provider", PROVIDER, "and model", MODEL, "in the UI. Logs: /content/dsh.log")

## Notes
- Qwen3.8-27B is a hybrid thinking model; the reasoning parser separates the thinking text from the answer.
- The model is picked from the GPU's memory (section 2). Large FP8 checkpoints are downloaded to Google Drive on the first run (about 31 GB for the 27B) and read from Drive afterwards, so the first load can be slow.
- The vLLM packages are cached on Drive (`MyDrive/llm_models/builds`) as wheels. Set `REFRESH_WHEELS = True` to update them. Install errors are in `/content/pip_install.log`; server errors in `/content/vllm.log`.
- A100 GPUs have no native FP8 math, so vLLM runs FP8 weights through a slower weight-only path there; L4 and G4 handle FP8 natively.
- vLLM's newest Blackwell (G4) support depends on the version pip installs; if the server fails to start on a G4, check `/content/vllm.log`.
- The tunnel and API key last only for this session. Run the tunnel cell again for a new URL after a restart.